In [ ]:
"""
=============================================================================
  Deepfake Detection — Preprocessing Pipeline
  Dataset : FaceForensics++ (C23) via metadata CSV
  Platform: Kaggle (also runs locally — adjust the paths below)

  For each video listed in the CSV:
    1. sample 1 frame every FRAME_STEP frames (max MAX_FRAMES)
    2. detect the best face with MTCNN (conf >= MIN_CONF, size >= MIN_FACE_PX)
    3. crop with MARGIN, resize to TARGET_SIZE (Lanczos), save as JPEG

  Output layout (consumed as-is by train.py):
    <WORKING>/preprocessed/<label>/<video_stem>/frame_XXXXX.jpg
    <WORKING>/dataset_index.csv          — frame index: frame_path,label,video_id,split
    <WORKING>/preprocess_progress.json   — resume checkpoint (every SAVE_EVERY videos)

  Re-running the script resumes from where it left off.

  To split the work across several Kaggle sessions, run one session per CSV
  slice using ROW_OFFSET / ROW_LIMIT (e.g. session A: offset 0, limit 3500;
  session B: offset 3500). train.py merges the resulting datasets.
=============================================================================
"""

import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
if ON_KAGGLE:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "mtcnn", "opencv-python-headless", "tqdm"])

import json
import random
import shutil
import warnings

import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_INPUT    = Path("/kaggle/input") if ON_KAGGLE else Path("data")
WORKING          = Path("/kaggle/working") if ON_KAGGLE else Path("output")
PREPROCESSED_DIR = WORKING / "preprocessed"
PROGRESS_FILE    = WORKING / "preprocess_progress.json"
INDEX_FILE       = WORKING / "dataset_index.csv"

CSV_FILENAME = "FF++_Metadata_Shuffled.csv"   # searched recursively under DATASET_INPUT

# ── CSV slice (for splitting work across Kaggle sessions) ────────────────────
ROW_OFFSET = 0        # skip this many data rows (e.g. 3500 for the second session)
ROW_LIMIT  = None     # process at most this many rows (None = all remaining)

# ── Preprocessing parameters ──────────────────────────────────────────────────
FRAME_STEP   = 10            # extract 1 frame every N frames (~3 fps at 30 fps)
MAX_FRAMES   = 30            # hard cap per video
TARGET_SIZE  = (224, 224)    # EfficientNet-B4 input resolution
MARGIN       = 0.20          # MTCNN bounding-box margin (20 %)
MIN_CONF     = 0.95          # MTCNN minimum confidence
MIN_FACE_PX  = 60            # minimum face width / height in pixels
SAVE_EVERY   = 10            # save progress checkpoint every N videos

# ── Split ratios (applied per class, at video level) ──────────────────────────
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}

# ── Optional quality filters ──────────────────────────────────────────────────
APPLY_GAUSSIAN = False   # light Gaussian blur to reduce MPEG blocking artefacts
APPLY_CLAHE    = False   # adaptive histogram equalisation for illumination

PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  LOAD & VALIDATE CSV
# ─────────────────────────────────────────────────────────────────────────────

def find_csv() -> Path:
    candidates = list(DATASET_INPUT.rglob(CSV_FILENAME))
    if not candidates:
        raise FileNotFoundError(
            f"{CSV_FILENAME} not found under {DATASET_INPUT}.\n"
            "Make sure you have added the dataset to this kernel."
        )
    if len(candidates) > 1:
        print(f"[WARN] Multiple CSV files found - using: {candidates[0]}")
    return candidates[0]

CSV_PATH = find_csv()
print(f"CSV : {CSV_PATH}")

skiprows = range(1, ROW_OFFSET + 1) if ROW_OFFSET else None
meta = pd.read_csv(CSV_PATH, skiprows=skiprows, nrows=ROW_LIMIT)
meta.columns = [c.strip() for c in meta.columns]   # strip accidental spaces

REQUIRED_COLS = {"File Path", "Label"}
missing = REQUIRED_COLS - set(meta.columns)
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}. Found: {list(meta.columns)}")

meta["label_norm"] = meta["Label"].str.strip().str.lower()

print(f"\nCSV loaded : {len(meta)} rows (offset={ROW_OFFSET}, limit={ROW_LIMIT})")
print("Label distribution:")
print(meta["label_norm"].value_counts().to_string())

# ─────────────────────────────────────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def resolve_path(raw: str) -> Path | None:
    """
    Try to locate the video file.
    1. Absolute path as-is.
    2. Relative to the CSV's parent dataset directory.
    3. Recursive search under DATASET_INPUT by filename.
    """
    p = Path(raw)
    if p.exists():
        return p
    alt = CSV_PATH.parent.parent / p
    if alt.exists():
        return alt
    matches = list(DATASET_INPUT.rglob(p.name))
    if matches:
        return matches[0]
    return None


def extract_frames(video_path: Path, output_dir: Path) -> list[Path]:
    """
    Extract up to MAX_FRAMES frames from video_path, one every FRAME_STEP.
    Saves raw JPEGs to output_dir and returns their paths.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    saved_paths, count, saved = [], 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % FRAME_STEP == 0:
            out = output_dir / f"frame_{saved:05d}.jpg"
            cv2.imwrite(str(out), frame)
            saved_paths.append(out)
            saved += 1
        count += 1
        if saved >= MAX_FRAMES:
            break

    cap.release()
    return saved_paths


def build_detector():
    try:
        from mtcnn import MTCNN
        det = MTCNN()
        print("[INFO] MTCNN detector ready.")
        return det
    except Exception as e:
        print(f"[WARN] MTCNN unavailable ({e}). Frames will be resized without face crop.")
        return None


def crop_face(img_bgr: np.ndarray, detector) -> np.ndarray | None:
    """
    Detect the highest-confidence face, apply margin, resize to TARGET_SIZE.
    Returns None if no valid face found (frame will be discarded).
    If detector is None, falls back to a plain resize.
    """
    if detector is None:
        return cv2.resize(img_bgr, TARGET_SIZE, interpolation=cv2.INTER_LANCZOS4)

    h, w = img_bgr.shape[:2]
    img_rgb    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    detections = detector.detect_faces(img_rgb)

    if not detections:
        return None

    best = max(detections, key=lambda d: d["confidence"])

    if best["confidence"] < MIN_CONF:
        return None

    x, y, bw, bh = best["box"]
    x, y = max(0, x), max(0, y)   # MTCNN can return negative coords
    if bw < MIN_FACE_PX or bh < MIN_FACE_PX:
        return None

    pad_x, pad_y = int(bw * MARGIN), int(bh * MARGIN)
    x1 = max(0, x - pad_x);      y1 = max(0, y - pad_y)
    x2 = min(w, x + bw + pad_x); y2 = min(h, y + bh + pad_y)
    face = img_bgr[y1:y2, x1:x2]
    if face.size == 0:
        return None

    return cv2.resize(face, TARGET_SIZE, interpolation=cv2.INTER_LANCZOS4)


def apply_gaussian_filter(img: np.ndarray,
                          kernel_size=(3, 3), sigma=0.5) -> np.ndarray:
    """Light Gaussian smoothing — reduces MPEG blocking artefacts."""
    return cv2.GaussianBlur(img, kernel_size, sigma)


def apply_clahe(img_bgr: np.ndarray) -> np.ndarray:
    """
    Adaptive histogram equalisation on the L channel (LAB space).
    Corrects heterogeneous illumination across frames.
    """
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_eq   = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)


def load_progress() -> dict:
    """Load saved progress from disk (empty dict if none exists)."""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            data = json.load(f)
        print(f"[RESUME] Progress file found - {len(data['processed_videos'])} videos already done.")
        return data
    return {"processed_videos": {}, "records": []}


def save_progress(progress: dict) -> None:
    """Persist progress to disk atomically."""
    tmp = PROGRESS_FILE.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(progress, f)
    tmp.replace(PROGRESS_FILE)

# ─────────────────────────────────────────────────────────────────────────────
#  MAIN PREPROCESSING LOOP
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  PREPROCESSING — Frame Extraction + Face Crop")
print("=" * 60)
print(f"  FRAME_STEP   = {FRAME_STEP}  (1 frame every {FRAME_STEP} frames)")
print(f"  MAX_FRAMES   = {MAX_FRAMES}  per video")
print(f"  TARGET_SIZE  = {TARGET_SIZE[0]}x{TARGET_SIZE[1]} px (Lanczos)")
print(f"  MTCNN conf  >= {MIN_CONF}  |  min face = {MIN_FACE_PX} px  |  margin = {int(MARGIN*100)}%")
print(f"  Gaussian     = {APPLY_GAUSSIAN}")
print(f"  CLAHE        = {APPLY_CLAHE}")
print(f"  Save every   = {SAVE_EVERY} videos")
print("=" * 60 + "\n")

detector = build_detector()
progress = load_progress()
processed_videos = progress["processed_videos"]   # {video_stem: {"kept": int, "rejected": int}}
records          = progress["records"]            # list of {frame_path, label, video_id}

video_paths = meta["File Path"].tolist()
labels      = meta["label_norm"].tolist()
total       = len(meta)
videos_this_run = 0   # counter for SAVE_EVERY logic

for idx, (raw_path, label) in enumerate(
        tqdm(zip(video_paths, labels), total=total, desc="Videos")):

    video_path = resolve_path(raw_path)

    if video_path is None:
        tqdm.write(f"[SKIP] Not found: {raw_path}")
        continue

    video_stem = f"{video_path.parent.name}_{video_path.stem}"
    video_id   = f"{label}/{video_stem}"

    # Already processed in a previous run → skip
    if video_stem in processed_videos:
        continue

    out_dir = PREPROCESSED_DIR / label / video_stem
    raw_dir = out_dir / "_raw"

    # Step 1: extract raw frames
    raw_frames = extract_frames(video_path, raw_dir)

    if not raw_frames:
        tqdm.write(f"[WARN] No frames extracted from {video_path.name}")
        processed_videos[video_stem] = {"kept": 0, "rejected": 0}
        videos_this_run += 1
        continue

    kept = 0
    rejected = 0

    for raw_frame_path in raw_frames:
        img_bgr = cv2.imread(str(raw_frame_path))
        if img_bgr is None:
            rejected += 1
            continue

        # Step 2: MTCNN face crop
        cropped = crop_face(img_bgr, detector)
        if cropped is None:
            rejected += 1
            continue

        # Step 3: optional quality filters
        if APPLY_GAUSSIAN:
            cropped = apply_gaussian_filter(cropped)
        if APPLY_CLAHE:
            cropped = apply_clahe(cropped)

        # Step 4: save final cropped face
        out_dir.mkdir(parents=True, exist_ok=True)
        final_path = out_dir / raw_frame_path.name
        cv2.imwrite(str(final_path), cropped)

        records.append({
            "frame_path": str(final_path),
            "label"     : label,
            "video_id"  : video_id,
        })
        kept += 1

    # Clean up raw frames
    if raw_dir.exists():
        shutil.rmtree(raw_dir)

    processed_videos[video_stem] = {"kept": kept, "rejected": rejected}
    videos_this_run += 1

    tqdm.write(
        f"[{idx+1:>5}/{total}] {video_path.name:<40} "
        f"kept={kept}  rejected={rejected}"
    )

    # Save checkpoint every SAVE_EVERY videos
    if videos_this_run % SAVE_EVERY == 0:
        progress["processed_videos"] = processed_videos
        progress["records"]          = records
        save_progress(progress)
        tqdm.write(f"  [OK] Progress saved ({len(processed_videos)} videos total, "
                   f"{len(records)} frames)")

# Final save (catches any remainder < SAVE_EVERY)
progress["processed_videos"] = processed_videos
progress["records"]          = records
save_progress(progress)

print(f"\nTotal videos processed : {len(processed_videos)}")
print(f"Total valid frames     : {len(records)}")

# ─────────────────────────────────────────────────────────────────────────────
#  BALANCE & SPLIT  (video-aware — frames of one video never cross splits)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  BALANCE & SPLIT  (video-aware)")
print("=" * 60)

frames_df = pd.DataFrame(records)

if frames_df.empty:
    raise RuntimeError(
        "No frames were saved. "
        "Check that 'File Path' in the CSV points to accessible video files."
    )

real_vids = sorted(frames_df.loc[frames_df["label"] == "real", "video_id"].unique())
fake_vids = sorted(frames_df.loc[frames_df["label"] == "fake", "video_id"].unique())
print(f"Unique videos — real: {len(real_vids)}  fake: {len(fake_vids)}")

# Under-sample at video level to equalise classes
min_vids = min(len(real_vids), len(fake_vids))
random.Random(SEED).shuffle(real_vids)
random.Random(SEED).shuffle(fake_vids)
real_vids = real_vids[:min_vids]
fake_vids = fake_vids[:min_vids]
print(f"After balance — {min_vids} videos per class")

def split_video_ids(vids: list) -> dict:
    n       = len(vids)
    n_train = int(n * SPLIT_RATIOS["train"])
    n_val   = int(n * SPLIT_RATIOS["val"])
    return {
        "train": set(vids[:n_train]),
        "val"  : set(vids[n_train:n_train + n_val]),
        "test" : set(vids[n_train + n_val:]),
    }

split_of_video = {}
for vids in [real_vids, fake_vids]:
    for split_name, vid_set in split_video_ids(vids).items():
        for vid in vid_set:
            split_of_video[vid] = split_name

balanced = frames_df[frames_df["video_id"].isin(split_of_video)].copy()
balanced["split"] = balanced["video_id"].map(split_of_video)
balanced = balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nSplit statistics:")
for split in ["train", "val", "test"]:
    sub = balanced[balanced["split"] == split]
    print(f"  {split:5s} | {len(sub):>6} frames | "
          f"real={(sub.label == 'real').sum():>5}  "
          f"fake={(sub.label == 'fake').sum():>5}  "
          f"videos={sub.video_id.nunique()}")

# ─────────────────────────────────────────────────────────────────────────────
#  SAVE FINAL INDEX
# ─────────────────────────────────────────────────────────────────────────────

balanced.to_csv(INDEX_FILE, index=False)
print(f"\nDataset index saved -> {INDEX_FILE}")
print("\nColumns:", list(balanced.columns))
print(balanced.head())

# ─────────────────────────────────────────────────────────────────────────────
#  SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

total_kept     = sum(v["kept"]     for v in processed_videos.values())
total_rejected = sum(v["rejected"] for v in processed_videos.values())
rejection_rate = total_rejected / (total_kept + total_rejected) * 100 if (total_kept + total_rejected) > 0 else 0

print("\n" + "=" * 60)
print("  PREPROCESSING SUMMARY")
print("=" * 60)
print(f"  Videos processed   : {len(processed_videos)}")
print(f"  Frames kept        : {total_kept}")
print(f"  Frames rejected    : {total_rejected}  ({rejection_rate:.1f}% — MTCNN filter)")
print(f"  Frames in index    : {len(balanced)}  (after balance)")
print(f"  Gaussian filter    : {'ON' if APPLY_GAUSSIAN else 'OFF'}")
print(f"  CLAHE              : {'ON' if APPLY_CLAHE else 'OFF'}")
print(f"  Output directory   : {PREPROCESSED_DIR}")
print(f"  Dataset index      : {INDEX_FILE}")
print(f"  Progress file      : {PROGRESS_FILE}")
print("=" * 60)
print("\nPreprocessing complete. Ready for train.py.")
